# 🤖 Лаборатория обучения RL-агента исполнения (Maskable PPO + L2 FIFO)
Этот блокнот позволяет в реальном времени запускать, отслеживать и препарировать обучение **RL-агента микроструктурного исполнения (`MaskableActorCritic`)** поверх 10-уровневого L2 FIFO-стакана SafeTrade и сигналов триады ансамблей (`H=5s, 12s, 25s`).

In [ ]:
# 📊 ЯЧЕЙКА 1: Обзор кэша полигона и кривой сходимости последнего обучения
import importlib
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import _tools.train_rl_execution as _rl_mod
importlib.reload(_rl_mod)
from _tools.train_rl_execution import build_or_load_rl_cache

TAIL_HOURS = 72.0
data = build_or_load_rl_cache(coin="prl", tail_hours=TAIL_HOURS)
print(f"✅ Размерность матрицы состояния рынка: {data['X_market'].shape} | Период: {data['ts'][0]} ➔ {data['ts'][-1]}")

hist_path = Path("models/prl/rl_execution/training_history.json")
if hist_path.exists():
    h_data = json.loads(hist_path.read_text())
    df_h = pd.DataFrame(h_data["history"])
    base = h_data["baseline"]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.2))
    ax1.axhline(0.0, color="gray", ls="--", alpha=0.6)
    ax1.axhline(base["pnl_usd"], color="crimson", ls=":", label=f"Эвристика (${base['pnl_usd']:+.2f}, {base['trades']} сд.)")
    ax1.plot(df_h["epoch"], df_h["pnl_usd"], marker="o", label="OOS PnL эпохи ($)")
    ax1.plot(df_h["epoch"], df_h["best_pnl_usd"], ls="--", color="forestgreen", lw=2, label="Рекорд OOS PnL ($)")
    ax1.set_title("Динамика обучения Maskable PPO на слепом OOS (25.2 ч)")
    ax1.set_xlabel("Эпоха PPO"); ax1.set_ylabel("Чистый PnL (USDT)"); ax1.legend(); ax1.grid(alpha=0.3)
    ax2.plot(df_h["epoch"], df_h["exp_edge_ticks"], marker="s", color="darkorange", label="Ожидаемый Edge входа (тики)")
    ax2.set_title("Избирательность агента: Ex-Ante Expected Edge")
    ax2.set_xlabel("Эпоха PPO"); ax2.set_ylabel("Тики"); ax2.legend(); ax2.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
    display(df_h)

In [ ]:
# 🚀 ЯЧЕЙКА 2: Запуск живого обучения RL-агента с онлайн-графиком после каждой эпохи
import importlib
import _tools.train_rl_execution as _rl_mod
importlib.reload(_rl_mod)

res = _rl_mod.train_rl_agent(
    coin="prl",
    tail_hours=72.0,
    ppo_epochs=25,
    steps_per_epoch=16384,
    live_plot=True,
)

In [ ]:
# 🔍 ЯЧЕЙКА 3: Детальный слепой OOS-экзамен на 72ч полигоне (22.10.2026 14:40:38 МСК -> 23.10.2026 15:52:37 МСК)
import importlib
import _tools.train_rl_execution as _rl_mod
importlib.reload(_rl_mod)

df_ppo_trades, df_base_trades = _rl_mod.inspect_and_plot_rl_oos(coin="prl", tail_hours=72.0)
print(f"\n🏆 Сделки лучшей политики Maskable PPO ({len(df_ppo_trades)} шт.):")
display(df_ppo_trades)
print(f"\n📉 Сделки базовой эвристики ({len(df_base_trades)} шт.):")
display(df_base_trades.head(10))

In [ ]:
# 🚀 ЯЧЕЙКА 4: Чистый Форвард-тест на свежих данных строго после бэктеста (23.10.2026 15:52:38 МСК -> 24.10.2026 12:12:40 МСК)
import importlib
import _tools.train_rl_execution as _rl_mod
importlib.reload(_rl_mod)

df_fwd_ppo, df_fwd_base = _rl_mod.inspect_and_plot_rl_oos(
    coin="prl",
    tail_hours=24.0,
    start_ts_msk="2026-10-23 15:52:37+03:00",
)
print(f"\n🏆 Сделки на чистом Форвард-тесте Maskable PPO v3 ({len(df_fwd_ppo)} шт.):")
display(df_fwd_ppo)
print(f"\n📉 Сделки базового эталона v2 ({len(df_fwd_base)} шт.):")
display(df_fwd_base.head(10))